# 3. Validate outputs and evaluate confirmed GT matches
Format validity and ID coverage are reported for every run, including test inference. Test currently has no labels.jsonl: its predictions remain available for inspection, and no quantitative role scores are calculated.

When GT is available, only unique exact normalized page-local matches or manually confirmed matches count. Character labels map to SPOKEN, NARRATION to NARRATED, and UNKNOWN is ignored. Unmatched detections are unscored, never assigned OTHER. First-attempt scores and retry recovery are reported separately.

The review CSV allows gt_index, confirmed, and is_thought edits. Mark is_thought only after review; it is not inferred from a generic SPOKEN-to-NARRATED error. Magi essential appears only in this analysis, never in the Qwen inputs. Dependencies: pandas and scikit-learn.


In [ ]:
from pathlib import Path
from collections import Counter
import json
import unicodedata
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from IPython.display import display

MAGI_ROOT = Path(r"C:\Users\ESHAAN\BYTE\Applied-Task\Magi")
ROOT = MAGI_ROOT / "TextRoleVL"
RUN_NAME = "qwen_tail_previous_prompt"  # same as notebook 2
OUTPUTS = ROOT / "outputs" / RUN_NAME
EVALUATION = ROOT / "evaluation" / RUN_NAME
EVALUATION.mkdir(parents=True, exist_ok=True)

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def normalize(text):
    text = unicodedata.normalize("NFKC", text).casefold()
    return "".join(c for c in text if c.isalnum())

def gt_role(speaker):
    if speaker == "NARRATION":
        return "NARRATED"
    if speaker.lower().startswith("character") and speaker[9:].isdigit():
        return "SPOKEN"
    if speaker == "UNKNOWN":
        return None
    raise ValueError(f"Unexpected GT speaker label: {speaker}")

run = read_json(OUTPUTS / "run.json")
gt_by_sequence = {}
for split in {entry["split"] for entry in run["sequences"]}:
    path = MAGI_ROOT / "TaskData" / split / "labels.jsonl"
    if not path.exists():
        print(f"No GT for {split}: format checks and prediction inspection only.")
        continue
    with path.open(encoding="utf-8") as file:
        for line in file:
            record = json.loads(line)
            gt_by_sequence[record["sequence_id"]] = record["pages"]

## Build a review table without predictions
The existing CSV is preserved on reruns. Inspect GT/OCR alignment before scoring. For unmatched rows, enter a page-local zero-based `gt_index` and set `confirmed` to True only after checking the transcript/page. Leave uncertain rows unconfirmed.

Set `is_thought` to True only after visual/transcript review confirms a character thought. Leave it blank otherwise. This annotation is analysis-only.

In [ ]:
rows = []
for entry in run["sequences"]:
    sid = entry["sequence_id"]
    gt_pages = gt_by_sequence.get(sid, [[], [], []])
    raw_pages = read_json(Path(entry["cache_dir"]) / "magi_raw.json")["data"]
    for text in entry["manifest"]["texts"]:
        page_index = text["page"] - 1
        text_index = int(text["id"].split(":T")[1]) - 1
        key = normalize(text["text"])
        candidates = [i for i, gt in enumerate(gt_pages[page_index])
                      if key and normalize(gt["text"]) == key]
        match_index = candidates[0] if len(candidates) == 1 else None
        gt = gt_pages[page_index][match_index] if match_index is not None else {}
        essential = raw_pages[page_index]["is_essential_text"][text_index]
        rows.append({"sequence_id": sid, "ID": text["id"], "page": text["page"],
                     "OCR": text["text"], "gt_index": match_index,
                     "GT text": gt.get("text", ""), "GT speaker": gt.get("speaker", ""),
                     "confirmed": match_index is not None,
                     "match method": "unique exact normalized" if match_index is not None else "review needed",
                     "Magi essential": essential, "is_thought": ""})
review_path = EVALUATION / "match_review.csv"
if not review_path.exists():
    pd.DataFrame(rows).to_csv(review_path, index=False, encoding="utf-8-sig")
review = pd.read_csv(review_path, keep_default_na=False)
print("Review file:", review_path)
display(review)
# Show GT page-local indices for checking or confirming matches.
for entry in run["sequences"]:
    print(entry["sequence_id"])
    display(pd.DataFrame([{"page": p + 1, "gt_index": i, **gt}
                          for p, page in enumerate(gt_by_sequence.get(entry["sequence_id"], [[], [], []]))
                          for i, gt in enumerate(page)]))

## Reload reviewed matches and validate raw output again
Edit the CSV if needed, then run from this cell onward. Only `gt_index`, `confirmed`, and `is_thought` are editable fields. Other columns are refreshed from source data.

ID coverage is a diagnostic even for failed output; it never authorizes partial scoring. JSON validity measures JSON parsing, while complete-valid output also requires the exact schema, all IDs exactly once, and valid roles.

In [ ]:
def as_bool(value):
    if str(value).strip().lower() in ("true", "1"):
        return True
    if str(value).strip().lower() in ("false", "0", ""):
        return False
    raise ValueError(f"Expected True/False, got {value!r}")

# Same strict validator as notebook 2, kept here so evaluation is standalone.

In [ ]:
def validate_response(raw, expected_ids):
    report = {"json_valid": False, "valid": False, "id_coverage": 0.0,
              "missing": list(expected_ids), "duplicated": [], "invented": [], "errors": []}
    def reject_constant(value):
        raise ValueError(f"Non-JSON constant: {value}")

    def unique_fields(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise ValueError(f"Duplicate JSON field: {key}")
            result[key] = value
        return result

    try:
        rows = json.loads(raw, parse_constant=reject_constant, object_pairs_hook=unique_fields)
        report["json_valid"] = True
    except ValueError as error:
        report["errors"].append(str(error))
        return None, report
    if not isinstance(rows, list):
        report["errors"].append("Top-level output must be an array.")
        return None, report
    ids = []
    for row in rows:
        if not isinstance(row, dict) or set(row) != {"id", "role"}:
            report["errors"].append(f"Invalid object: {row!r}")
            continue
        if not isinstance(row["id"], str):
            report["errors"].append("ID must be a string.")
            continue
        ids.append(row["id"])
        if row["role"] not in ("SPOKEN", "NARRATED", "OTHER"):
            report["errors"].append(f"Invalid role for {row['id']}: {row['role']!r}")
    counts = Counter(ids)
    expected = set(expected_ids)
    report["missing"] = sorted(expected - set(ids))
    report["invented"] = sorted(set(ids) - expected)
    report["duplicated"] = sorted(key for key, count in counts.items() if count > 1)
    report["id_coverage"] = len(expected & set(ids)) / len(expected) if expected else 1.0
    report["valid"] = not any(report[key] for key in ("errors", "missing", "invented", "duplicated"))
    return (rows if report["valid"] else None), report

In [ ]:
review = pd.read_csv(review_path, keep_default_na=False)
assert not review.duplicated(["sequence_id", "ID"]).any(), "Duplicate review IDs"
source = pd.DataFrame(rows).drop(columns=["gt_index", "confirmed", "is_thought"])
assert set(zip(review.sequence_id, review.ID)) == set(zip(source.sequence_id, source.ID)), "Review ID mismatch"
review = source.merge(review[["sequence_id", "ID", "gt_index", "confirmed", "is_thought"]],
                      on=["sequence_id", "ID"], validate="one_to_one")
matched = []
for row in review.to_dict("records"):
    row["GT"] = None
    row["confirmed"] = as_bool(row["confirmed"])
    row["is_thought"] = as_bool(row["is_thought"])
    if row["confirmed"]:
        assert row["sequence_id"] in gt_by_sequence, "No GT for this sequence; leave matches unconfirmed."
        index = float(row["gt_index"])
        assert index.is_integer() and index >= 0, "Invalid GT index"
        page_gt = gt_by_sequence[row["sequence_id"]][row["page"] - 1]
        assert index < len(page_gt), "GT index outside page transcript"
        gt = page_gt[int(index)]
        row["GT text"], row["GT speaker"] = gt["text"], gt["speaker"]
        row["GT"] = gt_role(gt["speaker"])
    else:
        row["GT text"], row["GT speaker"] = "", ""
    assert not row["is_thought"] or row["GT"] == "SPOKEN", "Thought annotations require confirmed SPOKEN GT"
    matched.append(row)

formats = []
predictions = {}
for entry in run["sequences"]:
    sid = entry["sequence_id"]
    expected = [t["id"] for t in entry["manifest"]["texts"]]
    for attempt in (1, 2):
        path = OUTPUTS / f"{sid}_attempt_{attempt}_raw.txt"
        if not path.exists():
            if attempt == 1:
                formats.append({"sequence_id": sid, "attempt": attempt, "status": "not run",
                                "json_valid": False, "valid": False, "id_coverage": 0.0})
            continue
        parsed, report = validate_response(path.read_text(encoding="utf-8"), expected)
        formats.append({"sequence_id": sid, "attempt": attempt,
                        "status": "valid" if report["valid"] else "format failure", **report})
        if report["valid"]:
            predictions[(sid, attempt)] = {r["id"]: r["role"] for r in parsed}

formats = pd.DataFrame(formats)
formats.to_csv(EVALUATION / "format_reports.csv", index=False)
display(formats)
for attempt, group in formats[formats.status != "not run"].groupby("attempt"):
    print(f"Attempt {attempt}: {len(group)} returned sequences; JSON validity {group.json_valid.mean():.1%}; "
          f"complete valid output {group.valid.mean():.1%}; mean ID coverage {group.id_coverage.mean():.1%}")
first = formats[formats.attempt == 1]
print("Prepared sequences:", len(first), "Not run:", (first.status == "not run").sum())
failed_ids = set(first.loc[first.status == "format failure", "sequence_id"])
recovered = [sid for sid in failed_ids if (sid, 2) in predictions]
print(f"Retry recovery: {len(recovered)} / {len(failed_ids)} first-attempt format failures")

## Scores and failure analysis
SPOKEN → NARRATED becomes `thought→narration` only for manually confirmed thoughts. Otherwise it is `speech/thought→narration`.

The first-attempt table includes format failures and unmatched detections for inspection, but they contribute no quantitative labels. The separate recovered report uses attempt 2 only where attempt 1 failed.

In [ ]:
def analysis_table(use_retry=False):
    result = []
    for row in matched:
        sid = row["sequence_id"]
        attempt = 1
        pred_map = predictions.get((sid, 1))
        if pred_map is None and use_retry and (sid, 2) in predictions:
            attempt, pred_map = 2, predictions[(sid, 2)]
        pred = pred_map[row["ID"]] if pred_map is not None else None
        gt = row["GT"]
        if pred is None:
            error = "format failure / not run"
        elif not row["confirmed"]:
            error = "unmatched — unscored"
        elif gt is None:
            error = "UNKNOWN — ignored"
        elif gt == pred:
            error = "—"
        elif gt == "SPOKEN" and pred == "NARRATED":
            error = "thought→narration" if row["is_thought"] else "speech/thought→narration"
        elif gt == "NARRATED" and pred == "SPOKEN":
            error = "narration→speech"
        else:
            error = f"{gt.lower()}→{pred.lower()}"
        result.append({**row, "Pred": pred, "attempt": attempt, "Error type": error,
                       "scored": gt is not None and pred is not None})
    return pd.DataFrame(result)

def report_scores(table, name):
    scored = table[table.scored]
    print(name, "—", len(scored), "confirmed boxes from", scored.sequence_id.nunique(), "valid sequences")
    if scored.empty:
        print("No scorable matches. No accuracy/F1 reported.")
        return
    labels = ["SPOKEN", "NARRATED"]
    precision, recall, f1, support = precision_recall_fscore_support(
        scored.GT, scored.Pred, labels=labels, zero_division=0
    )
    print("Overall accuracy:", round(accuracy_score(scored.GT, scored.Pred), 4))
    print("Macro F1 (two GT classes):", round(float(f1.mean()), 4))
    print("Zero-support class metrics are undefined; displayed as 0. Check support.")
    metrics = pd.DataFrame({"precision": precision, "recall": recall, "F1": f1, "support": support}, index=labels)
    display(metrics)
    matrix = pd.crosstab(scored.GT, scored.Pred).reindex(index=labels, columns=labels + ["OTHER"], fill_value=0)
    display(matrix)
    thoughts = scored[scored.is_thought]
    print("Reviewed thoughts:", len(thoughts))
    if len(thoughts):
        print("Thoughts kept SPOKEN:", f"{(thoughts.Pred == 'SPOKEN').mean():.1%}")
    metrics.to_csv(EVALUATION / f"{name}_metrics.csv")
    matrix.to_csv(EVALUATION / f"{name}_confusion.csv")

first_table = analysis_table()
first_table.to_csv(EVALUATION / "first_attempt_analysis.csv", index=False, encoding="utf-8-sig")
report_scores(first_table, "first_attempt")
columns = ["sequence_id", "ID", "OCR", "GT", "Pred", "Magi essential", "Error type"]
display(first_table[columns])
print("Errors only:")
display(first_table[first_table.scored & (first_table.GT != first_table.Pred)][columns])
if recovered:
    recovered_table = analysis_table(use_retry=True)
    recovered_table.to_csv(EVALUATION / "with_retry_analysis.csv", index=False, encoding="utf-8-sig")
    report_scores(recovered_table, "with_retry")